# MNIST MLP3 baseline comparison

This notebook validates and compares the persisted three-seed **SGD + Nesterov**, **AdamW**, and **Muon + auxiliary AdamW** controls. All uncertainty intervals use complete training runs as the unit of replication. Validation loss selects the reported best checkpoint; the official MNIST test set is monitoring-only.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir(): ROOT = candidate; break
    if (path / 'rg_baselines').is_dir(): ROOT = path; break
if ROOT is None: raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from rg_baselines.comparison import run_baseline_comparison
RUN_ROOT = Path(os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')).expanduser().resolve()
OUTPUT_DIR = RUN_ROOT / 'comparison'
print('run root:', RUN_ROOT)

In [ ]:
result = run_baseline_comparison(
    RUN_ROOT, output_dir=OUTPUT_DIR, show_plots=True
)
assert len(result.seeds) == 3
print('comparison outputs:', OUTPUT_DIR)

## Final and validation-selected performance

The validation-selected table evaluates the protected test set only after the checkpoint epoch has been chosen from validation loss.

In [ ]:
display(result.terminal_by_seed.sort_values(['checkpoint','optimizer_label','seed']))
display(result.terminal_summary.sort_values(['metric','checkpoint','optimizer_label']))
assert result.terminal_summary['n'].eq(3).all()

## Validation convergence and matched-seed contrasts

Convergence thresholds are defined on validation accuracy. Paired differences are reported for both final and validation-selected checkpoints.

In [ ]:
display(result.convergence_by_seed.sort_values(['optimizer_label','seed']))
display(result.convergence_summary.sort_values(['metric','optimizer_label']))
display(result.paired_final_differences.sort_values(['checkpoint','metric','contrast']))

## Layerwise WeightWatcher summary

`alpha`, `ERG_gap`, and randomized-MP `num_traps` are direct WeightWatcher outputs. Each layer is summarized across the three complete runs; layers are not treated as additional replicates.

In [ ]:
spectral = result.spectral_summary[result.spectral_summary['metric'].isin([
    'alpha','ERG_gap','num_traps','m_midpoint','trace_log_midpoint_per_eval'
])].sort_values(['metric','optimizer_label','layer','epoch'])
assert spectral['n'].eq(3).all()
display(spectral)

The historical result-directory key `sgd_momentum_muon` is retained only for compatibility. Its implementation and label are **Muon + auxiliary AdamW**. No test metric is used for learning-rate choice, checkpoint selection, or stopping.